## Author: Gabriel Vance
## Class: DSC550 - Data Mining
## Professor: Dr. Brett Werner
## Date: June 17, 2025

# Purpose: This notebook performs various tasks labeled "Exercises" for Week 8 of the class. Data must be present within the folder that this notebook is run in.
# This week focuses on Hyperparameter Tuning.
 

### Import libraries/packages

In [1]:
import pandas as pd
import numpy as np  
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV


### 1. Import the dataset and ensure that it loaded properly.

In [2]:
# Read data
df = pd.read_csv('Loan_Train.csv')
print(df.head())

    Loan_ID Gender Married Dependents     Education Self_Employed  \
0  LP001002   Male      No          0      Graduate            No   
1  LP001003   Male     Yes          1      Graduate            No   
2  LP001005   Male     Yes          0      Graduate           Yes   
3  LP001006   Male     Yes          0  Not Graduate            No   
4  LP001008   Male      No          0      Graduate            No   

   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0             5849                0.0         NaN             360.0   
1             4583             1508.0       128.0             360.0   
2             3000                0.0        66.0             360.0   
3             2583             2358.0       120.0             360.0   
4             6000                0.0       141.0             360.0   

   Credit_History Property_Area Loan_Status  
0             1.0         Urban           Y  
1             1.0         Rural           N  
2             1.0   

### 2. Prepare the data for modeling by performing the following steps:
### - Drop the column “Load_ID.”
### - Drop any rows with missing data.
### - Convert the categorical features into dummy variables.

In [3]:
# Drop the column “Load_ID.” | Drop any rows with missing data.
df_cleaned = df.drop('Loan_ID', axis=1).dropna()
# Convert the categorical features into dummy variables.
df_dummies = pd.get_dummies(df_cleaned, drop_first=True)
# Print the modified DataFrame with dummy variables
print(df_dummies.head())

   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
1             4583             1508.0       128.0             360.0   
2             3000                0.0        66.0             360.0   
3             2583             2358.0       120.0             360.0   
4             6000                0.0       141.0             360.0   
5             5417             4196.0       267.0             360.0   

   Credit_History  Gender_Male  Married_Yes  Dependents_1  Dependents_2  \
1             1.0         True         True          True         False   
2             1.0         True         True         False         False   
3             1.0         True         True         False         False   
4             1.0         True        False         False         False   
5             1.0         True         True         False          True   

   Dependents_3+  Education_Not Graduate  Self_Employed_Yes  \
1          False                   False              False

### 3. Split the data into a training and test set, where the “Loan_Status” column is the target.

In [4]:
# Features and target
X = df_dummies.drop('Loan_Status_Y', axis=1)
y = df_dummies['Loan_Status_Y']

# Split into train and test sets (default 75/25 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

### 4. Create a pipeline with a min-max scaler and a KNN classifier

In [5]:
# Create a pipeline with a min-max scaler and a KNN classifier
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('knn', KNeighborsClassifier())
])

### 5. Fit a default KNN classifier to the data with this pipeline. Report the model accuracy on the test set. Note: Fitting a pipeline model works just like fitting a regular model.

In [6]:
# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

# Report accuracy on the test set
test_accuracy = pipeline.score(X_test, y_test)
print(f"Test set accuracy: {test_accuracy:.4f}")

Test set accuracy: 0.7333


### 6. Create a search space for your KNN classifier where your “n_neighbors” parameter varies from 1 to 10.

In [7]:
# Define the search space for n_neighbors from 1 to 10
param_grid = {'knn__n_neighbors': list(range(1, 11))}
print(param_grid)

{'knn__n_neighbors': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]}


### 7. Fit a grid search with your pipeline, search space, and 5-fold cross-validation to find the best value for the “n_neighbors” parameter.

In [8]:

# Fit grid search with pipeline, param_grid, and 5-fold cross-validation
grid_search = GridSearchCV(pipeline, param_grid, cv=5)
grid_search.fit(X_train, y_train)

# Print best value for n_neighbors
print(f"Best n_neighbors: {grid_search.best_params_['knn__n_neighbors']}")
print(f"Best cross-validated score: {grid_search.best_score_:.4f}")

Best n_neighbors: 3
Best cross-validated score: 0.7722


### 8. Find the accuracy of the grid search best model on the test set.

In [9]:
# Find the accuracy of the grid search best model on the test set
test_accuracy_grid = grid_search.score(X_test, y_test)
print(f"Test set accuracy (GridSearch best model): {test_accuracy_grid:.4f}")

Test set accuracy (GridSearch best model): 0.7500


### 9. Now, repeat steps 6 and 7 with the same pipeline, but expand your search space to include logistic regression and random forest models with the hyperparameter values in section 12.3 of the Machine Learning with Python Cookbook.